In [1]:
import torch
import numpy as np
import json
# import scipy.special as sp

# import pickle as pkl
# import zlib
# import base64

/Users/aleksei/.local/share/virtualenvs/kutulu-_n6nfavE/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
# sys.path.append('/home/kutulu/projects/code-of-kutulu-client')
sys.path.append('..')

In [3]:
from src.envs.agents.reinforce_agent import REINFORCEAgent
from src.envs.agents.dqn_agent_ext import DQNAgentExt
from src.envs.agents.dqn_agent import DQNAgent

In [4]:
from src.envs.league.agent_description import AgentDescription
from experiments.run_experiment import get_agent_info
from src.envs.agents.agent_factory import get_agent

In [5]:
from src.envs.distance import find_path
from src.envs.kutulu_observer import KutuluClosestObserver, KutuluClosestExtObserver
from src.envs.kutulu_world import KutuluWorldEnv
from src.game.template import CELL_WALL, DEFAULT_KUTULU_ACTIONS, EXTENDED_KUTULU_ACTIONS
from src.game.template import MOVE_REL_POS, REL_POSITIONS
from src.envs.agent_validator import AgentValidator

In [6]:
def get_agent_by_params(competitor_type, competitor_config, new_experiment, legacy_encoder):
    agent_info = get_agent_info(competitor_config, new_experiment=new_experiment)
    agent_info['checkpoint_dir'] = agent_info['checkpoint_dir'].replace("/home/kutulu/projects", "/Users/aleksei/projects")
    checkpoint_dir = agent_info['checkpoint_dir']
    agent_info['legacy_encoder'] = legacy_encoder
    # agent_info['explicit_action_mask'] = [
    #     True,
    #     True,
    #     True,
    #     True,
    #     False,
    #     True,
    #     True,
    #     False,
    # ]
    agent = get_agent(agent_info)
    agent.train = True
    return agent

In [7]:
av = AgentValidator(EXTENDED_KUTULU_ACTIONS)
av_plan = AgentValidator(EXTENDED_KUTULU_ACTIONS, player_params=(100, 1, 0))

In [8]:
competitor_type, competitor_config, new_experiment = ('ppo', 'b1c35809c181481d90fab3e07e54bcf4', True)
legacy_encoder = False
agent = get_agent_by_params(competitor_type, competitor_config, new_experiment, legacy_encoder)
print(av.check_entity_nearby(agent, 'WANDERER', n_min=2, n_max=3, verbose=False))
print(av.check_entity_nearby(agent, 'SLASHER', n_min=2, n_max=3, verbose=False))
print(av.check_entity_nearby(agent, 'EXPLORER', n_min=2, n_max=3, verbose=False))

(1.0, 0.0, 4, 0.0, 0.0)
(1.0, 0.0, 1, 0.0, 0.0)
(1.0, 0.0, 4, 0.0, 0.0)


In [9]:
# av.check_entity_nearby(agent, 'EXPLORER', n_min=2, n_max=2, verbose=True, env_types=['normal', 'coridor', 'corner'])
av.check_entity_nearby(agent, 'EXPLORER', n_min=2, n_max=2, verbose=True, env_types=['normal'])

answer: {0}, action: 0, explorers: [(0, -2)], wanderers: []
action: True, std: 0.0
#########
#..#.#..#
#...1...#
#..#^#..#
#...0...#
#..#.#..#
#.......#
#..#.#..#
#########


answer: {1}, action: 1, explorers: [(2, 0)], wanderers: []
action: True, std: 0.0
#########
#..#.#..#
#.......#
#..#.#..#
#...0^1.#
#..#.#..#
#.......#
#..#.#..#
#########


answer: {2}, action: 2, explorers: [(0, 2)], wanderers: []
action: True, std: 0.0
#########
#..#.#..#
#.......#
#..#.#..#
#...0...#
#..#^#..#
#...1...#
#..#.#..#
#########


answer: {3}, action: 3, explorers: [(-2, 0)], wanderers: []
action: True, std: 0.0
#########
#..#.#..#
#.......#
#..#.#..#
#.1^0...#
#..#.#..#
#.......#
#..#.#..#
#########




([True, True, True, True], [0.0, 0.0, 0.0, 0.0], [0, 1, 2, 3])

In [10]:
self = av
entity_kind = 'EXPLORER'
# n_min, n_max = 

In [11]:
max_dist = len(self.normal_env.map) // 2
h, w = len(self.normal_env.map), len(self.normal_env.map[0])

In [12]:
explorers_list = []
for i in range(h):
    for j in range(w):
        x = self.player_pos[0] + i
        y = self.player_pos[1] + j
        if self.normal_env.map[i][j] != '#':
            explorers_list.append([(j - self.player_pos[1], i - self.player_pos[0])])

In [13]:
env = self.normal_env

In [14]:
action_list = []
output_list = []
for explorers in explorers_list:
    self._set_env(env, agent, explorers, wanderers=[], slashers=[])
    output = agent.inference_step(0)
    action = output['action']
    output_list.append(output)
    action_list.append(action)

In [15]:
_map = [list(x) for x in env.map]
for explorers, action in zip(explorers_list, action_list):
    explorer = explorers[0]
    x, y = self.player_pos[0] + explorer[0], self.player_pos[1] + explorer[1]
    _map[y][x] = str(action)
_map = [''.join(x) for x in _map]

In [16]:
_map

['#########',
 '#33#0#00#',
 '#3300000#',
 '#33#0#11#',
 '#3333111#',
 '#33#2#11#',
 '#2322222#',
 '#22#2#22#',
 '#########']

In [78]:
competitor_type, competitor_config, new_experiment = ('ppo', '05abe073de06428e896fcd880c9f3eac', True)
legacy_encoder = True
agent = get_agent_by_params(competitor_type, competitor_config, new_experiment, legacy_encoder)
print(av.check_entity_nearby(agent, 'WANDERER', n_min=2, n_max=3, verbose=False))
print(av.check_entity_nearby(agent, 'SLASHER', n_min=2, n_max=3, verbose=False))
print(av.check_entity_nearby(agent, 'EXPLORER', n_min=2, n_max=3, verbose=False))

(1.0, 0.0, 1, 0.0, 0.0)
(0.0, 0.0, 1, 0.0, 0.0)
(1.0, 0.0, 4, 0.0, 0.0)


In [79]:
competitor_type, competitor_config, new_experiment = ('qdn_conv', '20250622-045641', False)
legacy_encoder = False
agent = get_agent_by_params(competitor_type, competitor_config, new_experiment, legacy_encoder)
print(av.check_entity_nearby(agent, 'WANDERER', n_min=2, n_max=3, verbose=False))
print(av.check_entity_nearby(agent, 'SLASHER', n_min=2, n_max=3, verbose=False))
print(av.check_entity_nearby(agent, 'EXPLORER', n_min=2, n_max=3, verbose=False))

(1.0, 0.0, 1, 0.0, 0.0)
(0.875, 0.0, 1, 0.0, 0.0)
(1.0, 0.0, 4, 0.0, 0.0)
